# Basics: load, inspect, run and export an experiment

This notebook runs the demo configuration `data/bfi_demo_config.json` (four items of the
Big Five Inventory, answered by two personas) with a **real local Hugging Face model**.

**Requirements**

* Install the `huggingface` extra for local models and the `notebook` extra for progress
  bars in Jupyter:
  `pip install "rupsycho[huggingface,notebook] @ git+https://github.com/julianschelb/rupsycho.git"`
* The configuration uses `Qwen/Qwen2.5-0.5B-Instruct` on the CPU. It is downloaded from the
  Hugging Face Hub on first use (about 1 GB), and the eight generations take minutes at
  most on a laptop. The cells that load the model carry the `skip-execution` tag and have
  no stored output.
* You do not want to download anything? [quickstart.ipynb](quickstart.ipynb) walks through
  the same steps offline, with a scripted stand-in for the model.

In [1]:
import rupsycho as rup

## Load an experiment from a file

An experiment is described by one JSON file: the questionnaire, the personas, the models,
the prompt template and the seeds (see the
[configuration reference](https://julianschelb.github.io/rupsycho/configuration/)).
`experiment_from_file` validates the file and raises an error that names the offending
field if it is invalid. `experiment_from_dict` does the same for a dictionary, and
`experiments_from_files("configs/*.json")` loads many files at once.

In [2]:
from pathlib import Path

CONFIG = Path("data/bfi_demo_config.json")
if not CONFIG.exists():  # the notebook was opened from the repository root
    CONFIG = Path("examples") / CONFIG

experiment = rup.experiment_from_file(CONFIG)

In [3]:
print("experiment:", experiment.name)
for identifier, model in experiment.models.items():
    print(f"model     : {identifier!r} -> {model.type}, {model.name_or_path}")
print("personas  :", experiment.list_personas())
print("questions :", experiment.questionnaire.get_number_of_questions())
print("seeds     :", experiment.parameters.seeds)

experiment: Big Five Inventory example
model     : 'Qwen/Qwen2.5-0.5B-Instruct' -> local_huggingface, Qwen/Qwen2.5-0.5B-Instruct
personas  : ['Optimistic Persona', 'Conservative Persona']
questions : 5
seeds     : ['7']


The model is only *described* at this point. With `lazy_load_models` (the default) it is
loaded when the run reaches it and released afterwards, so a study with several large models
never holds more than one in memory.

Before spending compute, look at the exact prompt the model will receive:

In [4]:
experiment.print_assembled_prompt(item_idx=0, persona_idx=1)


++++++++++++++++++ assembled prompt (item 0, persona 1) ++++++++++++++++++
System: Objective: Answer as the following survey participant: Mr Grueber is 65 years old and very conservative with a reserved personality.
Here are a number of characteristics that may or may not apply to you. Please indicate the extent to which you agree or disagree with each statement by choosing one of the answer options.
Instructions: Choose from the list of answer options to answer the question. Answer the question using only the provided answer options. If none of the options are correct, choose the option that is closest to being correct. The solution must be provided in this format: {answer: "answer option"}
Human: Question: I see myself as someone who is talkative.
Answer Options: 1. Disagree strongly, 2. Disagree a little, 3. Neither agree nor disagree, 4. Agree a little, 5. Agree strongly
Answer:
+++++++++++++++++++++++++++++++++++++++++++++++++++++


## Run the experiment

`run()` asks every model every question as every persona, once per seed, and returns a
`RunSummary` (number of calls, failures, elapsed seconds). Useful arguments:

| Argument | Meaning |
| --- | --- |
| `callbacks=[...]` | receive every answer as soon as it is generated, see [Callbacks.ipynb](Callbacks.ipynb) |
| `on_error="warn"` / `"raise"` / `"ignore"` | what to do when a model call fails |
| `max_concurrency=N` | parallel calls for API models (local models run one after the other) |
| `show_progress=False` | hide the progress bar |
| `cumulative=True` | let each persona remember its earlier answers |

**This cell downloads and runs the model.**

In [ ]:
summary = experiment.run()
print(summary)

## Collect the results

`get_answers_as_dataframe()` returns one row per model x persona x seed x question. The
nested structure (question, model, persona, seed) is available from `get_answers()`.

In [ ]:
answers = experiment.get_answers_as_dataframe()
answers

## Save the results

The scratch folder `output/` next to this notebook is not part of the repository.
`export_to_file` writes the experiment configuration together with the answers as JSON
that `experiment_from_file` can load again (API keys are masked in exports); pass
`include_answers=False`, or use `experiment.to_config()`, to get the definition only.

In [ ]:
OUTPUT = CONFIG.parent.parent / "output"
OUTPUT.mkdir(exist_ok=True)

answers.to_csv(OUTPUT / "basics_answers.csv", index=False)
experiment.export_to_file(OUTPUT / "basics_experiment.json")

## Use a different model

Start from the configuration without its model and add any LangChain model, local or
hosted. The identifier labels the answers in the results.

```python
experiment = rup.load_example_experiment("bfi", models={})   # the bundled copy of the demo

# a model served by Ollama (pip install "rupsycho[ollama]")
from langchain_ollama import OllamaLLM

experiment.add_model(OllamaLLM(model="llama3.2"), identifier="llama3.2")

# a hosted model (pip install "rupsycho[openai]", needs OPENAI_API_KEY)
from langchain_openai import ChatOpenAI

experiment.add_model(ChatOpenAI(model="gpt-4o-mini"), identifier="gpt-4o-mini")

experiment.run(max_concurrency=8)   # hosted models can be called in parallel
```

`examples/api_models.py` collects configuration snippets for OpenAI, Ollama, Google and
DeepSeek, including the environment variables they read.